In [76]:
from InputData import *

instance_filename = "Construction_a1_o12_m3_an5_ar3_reduced.json"
#instance_filename = "Construction_a10_o107_m5_an57_ar12.json"
instance_filename = "Construction_a50_o578_m28_an276_ar66.json"

data = InputData(instance_filename)


In [77]:
# 2a. Sets
M = list()
W_m = dict()
N_m = dict()
for machine in data.machines:
    M.append(machine.id)
    W_m[machine.id] = [int(driver) for driver in machine.default_drivers]
    N_m[machine.id] = list()
    for orderItem in data.order_items:
        if orderItem.machine_type == machine.type:
            N_m[machine.id].append(orderItem.id)

W = list()
N_w = dict()
for worker in data.workers:
    W.append(worker.personal_number)
    for orderItem in data.order_items:
        if orderItem.worker_qualifications == []:
            if worker.personal_number not in N_w:
                N_w[worker.personal_number] = list()
            N_w[worker.personal_number].append(orderItem.id)
        elif orderItem.worker_qualifications == worker.qualifications:
            if worker.personal_number not in N_w:
                N_w[worker.personal_number] = list()
            N_w[worker.personal_number].append(orderItem.id)
        
    
N = list()
for orderItem in data.order_items:
    N.append(orderItem.id)

C = list()
N_c = dict()
for order in data.orders:
    C.append(order.site_number)
    N_c[order.site_number] = [int(item_id) for item_id in order.order_item_ids]


start_date = data.start_date
end_date = data.end_date

O_t = dict()  # Tag an dem der Auftrag startet
D_r = dict()  # Tagschichten mit Tag als Key
N_r = dict()  # Alle Schichten mit Tag als Key
O_t_start = dict()  # Startzeiten  
O_t_end = dict()  # Endzeiten
O_t_start_inverted = dict()  # Umgekehrtes O_t (Startzeiten)
O_t_end_inverted = dict()  # Umgekehrtes O_t_end (Endzeiten)

SECONDS_IN_A_DAY = 86400 # 

for orderItem in data.order_items:
    orderID = orderItem.id 

    # Startzeit
    orderItem_start_date = orderItem.start_time
    delta_start = (orderItem_start_date - start_date)
    t_start = delta_start.total_seconds() / SECONDS_IN_A_DAY
    t_start_int = int(t_start)


    # O_t: Gruppiert nach Tagen
    if t_start_int not in O_t:
        O_t[t_start_int] = []
    O_t[t_start_int].append(orderID)


    # D_r Tagschichten mit Tag als Key
    if t_start_int not in D_r:
        D_r[t_start_int] = []
    if orderItem.start_time.hour <= 12:
        D_r[t_start_int].append(orderID)
        
    # N_r Alle Schichten mit Tag als Key
    if t_start_int not in N_r:
        N_r[t_start_int] = []
    N_r[t_start_int].append(orderID)
    
    # Startzeit
    if t_start not in O_t_start:
        O_t_start[t_start] = []
    O_t_start[t_start].append(orderID)
    
    # Invertiertes Dictionary O_t_start_inverted
    O_t_start_inverted[orderID] = t_start

    # Endzeit
    orderItem_end_date = orderItem.end_time
    delta_end = (orderItem_end_date - start_date)
    t_end = delta_end.total_seconds() / SECONDS_IN_A_DAY

    # O_t_end: Gruppiert nach Endzeit
    if t_end not in O_t_end:
        O_t_end[t_end] = []
    O_t_end[t_end].append(orderID)
    
    # Invertiertes Dictionary O_t_end_inverted
    O_t_end_inverted[orderID] = t_end



P_mn = dict()
S_mn = dict()

d_ab = data.transport_routes
d_wb = data.work_routes
d_ij = list()
d_wj = list()

for i in data.order_items:
    row = []
    for j in data.order_items:
        a = next((k for k,v in N_c.items() if i.id in v))
        b = next((k for k,v in N_c.items() if j.id in v))
        row.append(d_ab[a][b])
    d_ij.append(row)


for i in data.workers:
    row = []
    for j in data.order_items:
        a = next((k for k,v in N_c.items() if j.id in v))
        row.append(d_wb[i.personal_number][a])
    d_wj.append(row)



SPEED = 1680 # Durchschnittliche Geschwindigkeit des Maschinentransports in 1680 km/Tag --> entspricht 70 km/h

start = len(N)
end = len(N) + 1

for m in M:
    for n in N_m[m]:

        if (m,start) not in P_mn:
            P_mn[m,start] = list()
            S_mn[m,start] = list()
            S_mn[m,start].append(end) # Anfügen Endknoten als Nachfolger des Startknotens
        if (m,end) not in P_mn:
            P_mn[m,end] = list()
            S_mn[m,end] = list()
            P_mn[m,end].append(start) # Anfügen Startknoten als Vorgänger des Endknotens


        P_mn[m,n] = list()
        S_mn[m,n] = list()


        P_mn[m,n].append(start) # Anfügen Startknoten als Vorgänger von n
        S_mn[m,start].append(n) # Anfügen n als Nachfolger des Startknotens

        P_mn[m,end].append(n) # Anfügen n als Vorgänger des Endknotens
        S_mn[m,n].append(end) # Anfügen Endknoten als Nachfolger von n
        
        for i in N_m[m]:
            if n != i:
                start_time_n = O_t_start_inverted[n]
                end_time_n = O_t_end_inverted[n]
                start_time_i = O_t_start_inverted[i]
                end_time_i = O_t_end_inverted[i]


                if start_time_n >= end_time_i + d_ij[i][n] / SPEED: 
                    P_mn[m,n].append(i)

                if start_time_i > end_time_n + d_ij[n][i] / SPEED:
                    S_mn[m,n].append(i)


P_wn = dict()
S_wn = dict()

P_time = 9/24 # 9 Stunden Pausenzeit zwischen zwei Schichten --> 9/24 Tage

for w in W:
    for n in N_w[w]:
        
        if (w,start) not in P_wn:
            P_wn[w,start] = list()
            S_wn[w,start] = list()
            S_wn[w,start].append(end)
        if (w,end) not in P_wn:
            P_wn[w,end] = list()
            S_wn[w,end] = list()
            P_wn[w,end].append(start)
        
        P_wn[w,n] = list()
        S_wn[w,n] = list()

        P_wn[w,n].append(start)
        S_wn[w,start].append(n)
        
        P_wn[w,end].append(n)
        S_wn[w,n].append(end)
        
        for i in N_w[w]:
            if n != i:
                start_time_n = O_t_start_inverted[n]
                end_time_n = O_t_end_inverted[n]
                start_time_i = O_t_start_inverted[i]
                end_time_i = O_t_end_inverted[i]

                if start_time_n >= end_time_i + P_time:
                    P_wn[w,n].append(i)

                if start_time_i >= end_time_n + P_time:
                    S_wn[w,n].append(i)
        



day_difference = end_date - start_date
T_range = list(range(day_difference.days + 1))


# 2b. Parameter

T = day_difference.days + 1

# T anpassen wenn Instanz nicht über den gesamten Zeitraum geht
end_date_adjusted = start_date
for orderItem in data.order_items:
    if end_date_adjusted < orderItem.start_time:
        end_date_adjusted = orderItem.start_time
T = (end_date_adjusted - start_date).days + 1




t_o = list()
for orderItem in data.order_items:
    t_o.append(orderItem.duration)

In [78]:
T

29

In [79]:
S_mn

{(0, 578): [579,
  0,
  1,
  2,
  3,
  4,
  5,
  6,
  7,
  8,
  9,
  10,
  11,
  12,
  13,
  14,
  15,
  16,
  17,
  18,
  19,
  20,
  21,
  22,
  23,
  24,
  25,
  26,
  27,
  28,
  29,
  30,
  31,
  32,
  33,
  34,
  35,
  36,
  37,
  38,
  39,
  40,
  41,
  42,
  43,
  44,
  45,
  46,
  47,
  48,
  49,
  50,
  51,
  52,
  53,
  54,
  55,
  56,
  57,
  58,
  59,
  60,
  61,
  62,
  63,
  64,
  65,
  66,
  67,
  68,
  69,
  70,
  71,
  72,
  73,
  74,
  75,
  76,
  77,
  78,
  79,
  80,
  81,
  82,
  83,
  84,
  85,
  86,
  87,
  88,
  89,
  90,
  91,
  92,
  93,
  94,
  95,
  96,
  97,
  98,
  99,
  100,
  101,
  102,
  103,
  104,
  105,
  106,
  107,
  108,
  109,
  110,
  111,
  112,
  113,
  114,
  115,
  116,
  117,
  118,
  119,
  120,
  121,
  122,
  123,
  124,
  125,
  126,
  127,
  128,
  129,
  130,
  131,
  132,
  133,
  134,
  135,
  136,
  137,
  138,
  139,
  140,
  141,
  142,
  143,
  144,
  145,
  146,
  147,
  148,
  149,
  150,
  151,
  152,
  153,
  154,
  155,
 

In [80]:
N = range(3,3+5+1)

In [81]:
print(*N)

3 4 5 6 7 8


In [82]:
T = range(3,3+14)

In [83]:
print(*T)

3 4 5 6 7 8 9 10 11 12 13 14 15 16
